# AKT 2b — neutralizacja ekspozycji i przeliczenie osi

## Po co

Ekstremy osi czystej (Akt 2) pokazały na dole klatki ciemne/prześwietlone → **ekspozycja to kandydat na nieusunięty konfound**. Sondy techniczne ("well-exposed") wysoko — spójne z tym podejrzeniem. Domykamy: mierzymy jasność i kontrast twardo, neutralizujemy razem z dziurką i ostrością (podprzestrzeń 4D), przeliczamy oś i sprawdzamy, czy "ludzka historia" przetrwa.

## Zasada: najpierw zmierz, potem usuwaj

Przed neutralizacją pipeline zmierzy d' jasności i kontrastu względem chosen/rejected. Usuwamy świadomie, ze zmierzonym uzasadnieniem.

## Optymalizacja czasu

Laplasjan z Drive zajął ~5h (I/O). Tu: **kopiujemy obrazy na lokalny dysk Colaba raz** (~10-20 min), potem liczymy jasność+kontrast lokalnie w jednym przejściu (minuty).


## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q open_clip_torch scikit-learn 2>/dev/null


In [ ]:
import json, os
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from PIL import Image
np.set_printoptions(precision=4, suppress=True)

DRIVE = Path('/content/drive/MyDrive/fsa_data')
PAIRS = DRIVE / 'fsa_pairs_filtered.csv'
EMB_CACHE = DRIVE / 'clip_embeddings.npz'
AXES = DRIVE / 'akt1_axes.npz'
IMAGES_DRIVE = DRIVE / 'fsa_images'
IMAGES_LOCAL = Path('/content/fsa_images_local')
EXPO_CACHE = DRIVE / 'exposure_cache.npz'   # jasność + kontrast

if not os.path.exists('confound_pipeline.py'):
    from google.colab import files
    print('Wgraj confound_pipeline.py:'); files.upload()
from confound_pipeline import ConfoundController, ConfoundDirection
print('✅ Setup OK')


## 1. Wczytaj pary, embeddingi, osie konfoundów z Aktu 1

In [ ]:
pairs = pd.read_csv(PAIRS, dtype={'similarity': float}, keep_default_na=False)
with np.load(EMB_CACHE, allow_pickle=True) as data:
    all_files = list(data['files']); all_emb = data['emb']
file_to_idx = {f: i for i, f in enumerate(all_files)}

with np.load(AXES) as d:
    v_hole, v_sharp = d['v_hole'], d['v_sharp']
    v_clean2 = d['v_clean']   # oś z Aktu 1 (2 konfoundy) do porównania

ci = pairs['chosen_file'].map(lambda f: file_to_idx.get(f,-1)).values
ri = pairs['rejected_file'].map(lambda f: file_to_idx.get(f,-1)).values
ok = (ci>=0)&(ri>=0)
pairs = pairs[ok].reset_index(drop=True); ci, ri = ci[ok], ri[ok]
E_c = all_emb[ci].astype(np.float64); E_r = all_emb[ri].astype(np.float64)
print(f'Pary: {len(pairs)}, embeddingi: {all_emb.shape}')

# Ten sam split co Akt 1 (seed 42) — porównywalność wyników
rng = np.random.RandomState(42)
perm = rng.permutation(len(pairs))
n_train = int(0.7*len(pairs))
tr_idx, te_idx = perm[:n_train], perm[n_train:]
print(f'Split (seed 42, jak Akt 1): train {len(tr_idx)}, test {len(te_idx)}')


## 2. Kopiuj obrazy lokalnie (raz) i policz jasność+kontrast (z cache)


In [ ]:
needed = pd.unique(pairs[['chosen_file','rejected_file']].values.ravel())

# Wczytaj cache jeśli jest
expo = {}
if EXPO_CACHE.exists():
    with np.load(EXPO_CACHE, allow_pickle=True) as d:
        expo = {f: (b, c) for f, b, c in zip(d['files'], d['bright'], d['contrast'])}
    print(f'Cache ekspozycji: {len(expo)}')
todo = [f for f in needed if f not in expo]
print(f'Do policzenia: {len(todo)}')

if todo:
    # Kopiuj folder na lokalny dysk (10-20 min raz; potem przetwarzanie w minuty)
    if not IMAGES_LOCAL.exists():
        print('Kopiuję obrazy na lokalny dysk Colaba (raz)...')
        !rsync -a --info=progress2 "{IMAGES_DRIVE}/" "{IMAGES_LOCAL}/"
    src_dir = IMAGES_LOCAL if IMAGES_LOCAL.exists() else IMAGES_DRIVE
    print(f'Liczę z: {src_dir}')

    def brightness_contrast(path):
        # IMREAD_REDUCED_GRAYSCALE_4: dekoduje JPEG w 1/4 rozdzielczości — szybciej,
        # a średnia/odchylenie jasności praktycznie identyczne
        img = cv2.imread(str(path), cv2.IMREAD_REDUCED_GRAYSCALE_4)
        if img is None: return (np.nan, np.nan)
        h, w = img.shape
        my, mx = int(0.12*h), int(0.12*w)   # bez ramki filmu
        crop = img[my:h-my, mx:w-mx].astype(np.float32)
        return (float(crop.mean()), float(crop.std()))

    from tqdm.auto import tqdm
    for f in tqdm(todo, desc='jasność+kontrast'):
        expo[f] = brightness_contrast(src_dir / f)
    files_arr = np.array(list(expo.keys()))
    np.savez_compressed(EXPO_CACHE, files=files_arr,
                        bright=np.array([expo[f][0] for f in files_arr]),
                        contrast=np.array([expo[f][1] for f in files_arr]))
    print('✅ Cache zapisany')

bright_c = pairs['chosen_file'].map(lambda f: expo.get(f,(np.nan,np.nan))[0]).values
bright_r = pairs['rejected_file'].map(lambda f: expo.get(f,(np.nan,np.nan))[0]).values
print(f'\n=== TWARDY TEST EKSPOZYCJI ===')
print(f'Mediana jasności chosen:   {np.nanmedian(bright_c):.1f}')
print(f'Mediana jasności rejected: {np.nanmedian(bright_r):.1f}')
mid = np.nanmedian(np.concatenate([bright_c, bright_r]))
dev_c = np.abs(bright_c - mid); dev_r = np.abs(bright_r - mid)
print(f'Frakcja par gdzie chosen bliżej środka ekspozycji: {np.nanmean(dev_c < dev_r):.2%}')
print('→ >50% = chosen lepiej naświetlone = ekspozycja jest konfoundem')


## 3. Zbuduj kierunki jasności i kontrastu + ZMIERZ wszystkie 4 konfoundy

In [ ]:
ctrl = ConfoundController()   # embeddery niepotrzebne: from_labels + gotowe wektory

# Wstrzyknij kierunki z Aktu 1
ctrl.directions.append(ConfoundDirection('hole_punch', v_hole, 'detector'))
ctrl.directions.append(ConfoundDirection('sharpness', v_sharp, 'labels'))

# Jasność: kwartyle na wszystkich plikach z embeddingami
vals_b = np.array([expo.get(f,(np.nan,np.nan))[0] for f in all_files])
vals_k = np.array([expo.get(f,(np.nan,np.nan))[1] for f in all_files])
vb = ~np.isnan(vals_b)
qb_lo, qb_hi = np.nanpercentile(vals_b, [25, 75])
cd_bright = ctrl.from_labels('brightness',
    all_emb[vb & (vals_b >= qb_hi)].astype(np.float64),
    all_emb[vb & (vals_b <= qb_lo)].astype(np.float64))
qk_lo, qk_hi = np.nanpercentile(vals_k, [25, 75])
cd_contr = ctrl.from_labels('contrast',
    all_emb[vb & (vals_k >= qk_hi)].astype(np.float64),
    all_emb[vb & (vals_k <= qk_lo)].astype(np.float64))
print(f'Zbudowano: {cd_bright}, {cd_contr}\n')

# POMIAR: d' każdego konfoundu względem chosen/rejected (zasada: mierz przed usuwaniem)
E_all = np.vstack([E_c, E_r])
labels = np.array([1]*len(E_c) + [0]*len(E_r))
print('Siła konfoundów względem etykiety chosen/rejected:')
for m in ctrl.measure_all(E_all, labels):
    flag = '⚠ GROŹNY' if m['dprime'] > 0.15 else 'słaby'
    print(f"  {m['name']:12} d'={m['dprime']:.3f}  corr={m['corr']:+.3f}  [{flag}]")
print('\n(usuwamy wszystkie 4 — dziurka i ostrość dla ciągłości z Aktem 1,')
print(' jasność i kontrast wg pomiaru powyżej)')


## 4. Neutralizacja 4D + przeliczenie osi (te same miary co Akt 1)

In [ ]:
from sklearn.linear_model import LogisticRegression

E_c3 = ctrl.neutralize(E_c)
E_r3 = ctrl.neutralize(E_r)
ver = ctrl.verify(E_all)
print(f'Neutralizacja 4 kierunków: maks. rzut = {ver["max_residual"]:.2e}, podprzestrzeń {ver["subspace_dim"]}D\n')

def dprime_paired(a, b):
    return abs(a.mean()-b.mean())/(np.sqrt(0.5*(a.var()+b.var()))+1e-12)

def find_axis_and_measure(Ec, Er, tr, te, label=''):
    v = Ec[tr].mean(0) - Er[tr].mean(0); v /= (np.linalg.norm(v)+1e-12)
    d_test = dprime_paired(Ec[te]@v, Er[te]@v)
    X_tr = np.vstack([Ec[tr],Er[tr]]); y_tr = np.array([1]*len(tr)+[0]*len(tr))
    X_te = np.vstack([Ec[te],Er[te]]); y_te = np.array([1]*len(te)+[0]*len(te))
    probe = LogisticRegression(max_iter=2000).fit(X_tr, y_tr)
    acc = probe.score(X_te, y_te)
    rng2 = np.random.RandomState(0); d_perm=[]
    for _ in range(20):
        sw = rng2.rand(len(tr))<0.5
        vp = np.where(sw[:,None],Er[tr],Ec[tr]).mean(0)-np.where(sw[:,None],Ec[tr],Er[tr]).mean(0)
        vp/= (np.linalg.norm(vp)+1e-12)
        d_perm.append(dprime_paired(Ec[te]@vp, Er[te]@vp))
    print(f'[{label}]')
    print(f"  d'={d_test:.3f} (szum {np.mean(d_perm):.3f})  probe acc={acc:.3f}")
    return {'v': v, 'dprime': d_test, 'acc': acc, 'noise': float(np.mean(d_perm))}

res3 = find_axis_and_measure(E_c3, E_r3, tr_idx, te_idx, 'OŚ v3 (4 konfoundy usunięte)')
v_clean3 = res3['v']
print(f'\ncos(oś_Akt1, oś_v3) = {np.dot(v_clean2, v_clean3):.3f}  (jak bardzo ekspozycja zmieniła oś)')


## 5. Gradient per sim_level — czy sygnatura treściowa przetrwała

In [ ]:
te_mask = np.zeros(len(pairs), bool); te_mask[te_idx] = True
print('Sygnał per poziom (oś v3, test):')
res_lvl = {}
for lvl in ['mikroruch','bliski','wyrazny']:
    sel = (pairs['sim_level']==lvl).values & te_mask
    d = dprime_paired(E_c3[sel]@v_clean3, E_r3[sel]@v_clean3)
    res_lvl[lvl] = d
    print(f'  {lvl:10}: d\'={d:.3f}  (n={sel.sum()})')
delta3 = (E_c3@v_clean3) - (E_r3@v_clean3)
print(f'\nAnaliza parowana: frakcja par chosen>rejected = {np.mean(delta3[te_mask]>0):.2%} (test)')


## 6. Sondy na osi v3 — czy 'ludzka historia' wciąż wygrywa

In [ ]:
import torch, open_clip
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model, _, _ = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
tokenizer = open_clip.get_tokenizer('ViT-B-32')
model = model.to(device).eval()

@torch.no_grad()
def text_embed(p):
    t = model.encode_text(tokenizer([p]).to(device))
    t = t/t.norm(dim=-1, keepdim=True)
    return t[0].cpu().numpy().astype(np.float64)

PROBES = {
  'CZASOWE': [("a photograph capturing the decisive moment","a photograph of an ordinary moment"),
    ("a person caught mid-gesture","a person standing still"),("the peak of the action","before the action begins"),
    ("a fleeting spontaneous instant","a static posed scene"),("candid movement frozen in time","a motionless posed subject")],
  'KOMPOZYCYJNE': [("a perfectly composed photograph","a poorly composed photograph"),
    ("balanced framing with strong visual structure","awkward framing with cluttered structure"),
    ("an elegant geometric composition","a chaotic random arrangement"),
    ("a well-framed subject with clear focal point","a badly cropped subject with no focal point"),
    ("harmonious visual balance in the frame","unbalanced distracting composition")],
  'TECHNICZNE': [("a sharp well-exposed photograph","a blurry poorly exposed photograph"),
    ("a high quality professional photograph","a low quality amateur snapshot"),
    ("a clean undamaged photograph","a damaged photograph with marks and scratches"),
    ("a crisp detailed image","a soft out-of-focus image")],
  'EKSPRESYJNE': [("a powerful emotional expression","a blank neutral expression"),
    ("an intimate human connection","a distant detached scene"),("a compelling human story","an uneventful empty scene"),
    ("an expressive face full of feeling","an expressionless face"),("people interacting with each other","people ignoring each other")],
}
rows=[]
for cat, plist in PROBES.items():
    for pos, neg in plist:
        pv = text_embed(pos)-text_embed(neg); pv/= (np.linalg.norm(pv)+1e-12)
        rows.append({'kategoria':cat,'sonda':pos[:44],'cos':float(np.dot(v_clean3,pv))})
df3 = pd.DataFrame(rows)
dfs = df3.reindex(df3['cos'].abs().sort_values(ascending=False).index)
print('Top 8 sond dla osi v3:')
for _, r in dfs.head(8).iterrows():
    print(f"  {r['cos']:+.4f}  [{r['kategoria'][:5]}] {r['sonda']}")
agg3 = df3.groupby('kategoria')['cos'].apply(lambda x: x.abs().mean()).sort_values(ascending=False)
print('\nAgregacja |cos| per kategoria (v3):')
print(agg3.to_string())


## 7. Ekstremy osi v3 — czy ciemne klatki zniknęły z dołu

In [ ]:
used = pd.unique(pairs[['chosen_file','rejected_file']].values.ravel())
idxs = np.array([file_to_idx[f] for f in used if f in file_to_idx])
names = np.array([f for f in used if f in file_to_idx])
# rzut na v3 wymaga zneutralizowanych embeddingów tych obrazów
emb_used = ctrl.neutralize(all_emb[idxs].astype(np.float64))
proj = emb_used @ v_clean3
order = np.argsort(proj); N=8
src = IMAGES_LOCAL if IMAGES_LOCAL.exists() else IMAGES_DRIVE
fig, axes = plt.subplots(2, N, figsize=(2.2*N, 5.5))
for j, oi in enumerate(order[-N:][::-1]):
    axes[0,j].imshow(Image.open(src/names[oi])); axes[0,j].axis('off')
    axes[0,j].set_title(f'+{proj[oi]:.2f}', fontsize=8, color='green')
for j, oi in enumerate(order[:N]):
    axes[1,j].imshow(Image.open(src/names[oi])); axes[1,j].axis('off')
    axes[1,j].set_title(f'{proj[oi]:.2f}', fontsize=8, color='red')
plt.suptitle('Ekstremy osi v3 (po neutralizacji ekspozycji)', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()
print('Porównaj z Aktem 2: czy dół to wciąż puste sceny, ale już NIE ciemne/prześwietlone?')


## 8. Podsumowanie i zapis

In [ ]:
print('='*60)
print('PORÓWNANIE OSI')
print('='*60)
print('Akt 1 (2 konfoundy: dziurka+ostrość):  d\'=0.195, acc=0.564')
print(f"Akt 2b (4 konfoundy: +jasność+kontrast): d'={res3['dprime']:.3f}, acc={res3['acc']:.3f}")
print(f"cos(oś_v2, oś_v3) = {np.dot(v_clean2, v_clean3):.3f}")
print(f"Gradient v3: mikroruch {res_lvl['mikroruch']:.3f} → bliski {res_lvl['bliski']:.3f} → wyrazny {res_lvl['wyrazny']:.3f}")
print(f"Wygrana kategoria sond (v3): {agg3.index[0]}")

np.savez_compressed(DRIVE/'akt2b_axes.npz', v_clean3=v_clean3,
    v_bright=cd_bright.vector, v_contrast=cd_contr.vector)
out = {'dprime_v3': res3['dprime'], 'acc_v3': res3['acc'], 'noise': res3['noise'],
       'per_level': res_lvl, 'cos_v2_v3': float(np.dot(v_clean2,v_clean3)),
       'agg_probes_v3': agg3.to_dict()}
with open(DRIVE/'akt2b_results.json','w') as f:
    json.dump(out, f, indent=2, default=float)
print('\n✅ Zapisano akt2b_axes.npz + akt2b_results.json')
